# Null models: 20 degree-preserving shuffles per network

## 1. Load the data
Same loader as in the statistics notebook.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx

NETWORKS = ['S1', 'S2', 'M1', 'M2', 'L1', 'L2']
DATA = Path('data') if Path('data').exists() else Path('../data')


def load_network(name):
    '''Reads nodes.csv and edges.csv of one company into an undirected graph without duplicates or self-loops.'''
    folder = DATA / f'{name}.csv'
    nodes = pd.read_csv(folder / 'nodes.csv', skipinitialspace=True)
    edges = pd.read_csv(folder / 'edges.csv', skipinitialspace=True)
    G = nx.Graph()
    G.add_nodes_from(nodes.iloc[:, 0])
    G.add_edges_from(zip(edges.iloc[:, 0], edges.iloc[:, 1]))
    G.remove_edges_from(list(nx.selfloop_edges(G)))
    return G


graphs = {name: load_network(name) for name in NETWORKS}

## 2. Generate the null models
Each null model is a copy of the network with its edge ends swapped 10 × E times: every node keeps its degree, but who is connected to whom becomes random. Seeds 0–19 make the models identical for everyone in the group. They are saved to `data/null_models/`.

In [ ]:
N_NULLS = 20
NULL_DIR = DATA / 'null_models'
NULL_DIR.mkdir(exist_ok=True)


def make_null(G, seed):
    '''Degree-preserving shuffle: swaps edge ends 10 x E times, so every node keeps its degree.'''
    R = G.copy()
    E = R.number_of_edges()
    nx.double_edge_swap(R, nswap=10 * E, max_tries=100 * E, seed=seed)
    return R


def null_path(name, i):
    '''File of null model number i of one network.'''
    return NULL_DIR / f'{name}_null_{i:02d}.edgelist.gz'


for name, G in graphs.items():
    start = time.time()
    for i in range(N_NULLS):
        path = null_path(name, i)
        if not path.exists():                          # skip models that were already generated
            nx.write_edgelist(make_null(G, seed=i), path, data=False)
    print(f'{name}: {N_NULLS} null models ready ({time.time() - start:.0f} s)')

## 3. Check the null models
Every null model must have the same number of edges and the same degree for every node as the real network. *Original edges kept* = share of real friendships still in place; low means well shuffled.

In [ ]:
def null_graphs(name):
    '''Loads the null models of one network one at a time (keeps memory low).'''
    for i in range(N_NULLS):
        yield nx.read_edgelist(null_path(name, i), nodetype=int)


def check_null(G, R):
    '''Compares one null model R with the real graph G.'''
    original = {tuple(sorted(e)) for e in G.edges()}
    kept = sum(tuple(sorted(e)) in original for e in R.edges())
    return {
        'same number of edges': R.number_of_edges() == G.number_of_edges(),
        'same degrees': dict(R.degree()) == dict(G.degree()),
        'original edges kept': kept / G.number_of_edges(),
    }


checks = {}
for name, G in graphs.items():
    results = pd.DataFrame([check_null(G, R) for R in null_graphs(name)])
    checks[name] = {
        'all same number of edges': results['same number of edges'].all(),
        'all same degrees': results['same degrees'].all(),
        'original edges kept (average)': results['original edges kept'].mean(),
    }
pd.DataFrame.from_dict(checks, orient='index').round(3)

## 4. Real vs null: clustering
Clustering is not kept fixed in the null models, so this is the test from the book (ch. 19): z = (real − null mean) / null std. A large z means the real clustering cannot be explained by the degrees alone.

In [ ]:
rows = {}
for name, G in graphs.items():
    null_values = [nx.transitivity(R) for R in null_graphs(name)]
    real = nx.transitivity(G)
    mean, std = np.mean(null_values), np.std(null_values)
    rows[name] = {'real': real, 'null mean': mean, 'null std': std,
                  'z-score': (real - mean) / std, 'real / null': real / mean}
    print(name, 'done')

pd.DataFrame.from_dict(rows, orient='index').round(3)

## 5. Using the null models in later notebooks

In [ ]:
NULL_DIR = DATA / 'null_models'
N_NULLS = 20


def null_graphs(name):
    '''Loads the null models of one network one at a time.'''
    for i in range(N_NULLS):
        yield nx.read_edgelist(NULL_DIR / f'{name}_null_{i:02d}.edgelist.gz', nodetype=int)